# Задание 2. Адаптивная реконструкция поверхностей по облакам точек ТЛО

**Этапы обработки:**
1. Чтение облака из `.ply`
2. Предобработка — фильтрация шума и нормализация
3. Разбиение на сегменты **по заранее заданному `label`**
4. Геометрический анализ сегментов (плотность, форма, кривизна, нормали, связность)
5. Отнесение сегмента к типу: `plane` / `tube` / `sphere` / `complex`
6. Подбор алгоритма реконструкции под тип
7. Реконструкция (Poisson / Alpha Shape / Ball Pivoting)
8. Объединение в единую модель
9. Оценка качества (RMSE, Hausdorff, артефакты, связность mesh)

## 1. Импорты и пути

In [ ]:
%pip install open3d numpy matplotlib scikit-learn scipy plyfile tqdm

In [ ]:
import json
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import KDTree, cKDTree
from plyfile import PlyData
from tqdm import tqdm

warnings.filterwarnings("ignore")

# исходные облака и место для результатов (macOS)
CLOUDS_DIR = Path("~/Datasets/lidar_pipe_fittings").expanduser()
RESULT_DIR = Path("outputs_task2")
RESULT_DIR.mkdir(exist_ok=True)

MESHES_DIR = RESULT_DIR / "meshes"
MESHES_DIR.mkdir(exist_ok=True)

n_found = len(list(CLOUDS_DIR.glob("*.ply"))) if CLOUDS_DIR.exists() else None
print("Найдено .ply:", n_found if n_found is not None else "директория не найдена")

## 2. Чтение и предобработка облака

In [ ]:
def read_ply_cloud(path):
    '''Читает PLY с полями x, y, z, scalar_Label (через plyfile).'''
    data = PlyData.read(str(path))
    el = data.elements[0].data
    coords = np.stack([el["x"], el["y"], el["z"]], axis=1).astype(np.float64)
    labels = np.asarray(el["scalar_Label"]).astype(np.int64)
    return coords, labels


def clean_and_normalize(coords, labels, nb_neighbors=20, std_ratio=2.0):
    '''Убирает статистические выбросы, центрирует и вписывает в единичную сферу.

    Метки переиндексируются согласованно с оставшимися точками.'''
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(coords)
    pcd, keep = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)

    coords_kept = np.asarray(pcd.points)
    labels_kept = labels[keep]

    center = coords_kept.mean(axis=0)
    coords_kept -= center
    radius = np.linalg.norm(coords_kept, axis=1).max()
    if radius > 0:
        coords_kept /= radius
    return coords_kept, labels_kept, center, radius

## 3. Разбиение на сегменты по `label`

In [ ]:
def segments_by_label(coords, labels, min_points=50):
    '''Группирует точки по значению label; слишком мелкие группы отбрасывает.'''
    result = {}
    for lbl in np.unique(labels):
        group = coords[labels == lbl]
        if len(group) >= min_points:
            result[int(lbl)] = group
    return result

## 4. Геометрический анализ сегмента

Признаки строятся на собственных значениях ковариационной матрицы (PCA) и kNN-окрестностях.
Linearity / Planarity / Sphericity (Demantke et al., 2011) — классический набор дескрипторов для облаков точек.

In [ ]:
def extract_geometry_features(seg):
    '''Геометрические дескрипторы сегмента.

    Возвращает linearity, planarity, sphericity, плотность, кривизну,
    согласованность нормалей и число связных компонент.'''
    feats = {}

    # PCA по всем точкам сегмента
    cov = np.cov(seg.T)
    lam = np.sort(np.linalg.eigvalsh(cov))[::-1]          # lam1 >= lam2 >= lam3
    lam = np.maximum(lam, 1e-12)
    linearity = (lam[0] - lam[1]) / lam[0]
    planarity = (lam[1] - lam[2]) / lam[0]
    sphericity = lam[2] / lam[0]
    feats.update(linearity=linearity, planarity=planarity,
                 sphericity=sphericity, eigvals=lam)

    # Плотность: среднее расстояние до ближайших соседей
    k = min(6, len(seg))
    if k >= 2:
        nbrs = NearestNeighbors(n_neighbors=k).fit(seg)
        dists, _ = nbrs.kneighbors(seg)
        feats["density"] = float(dists[:, 1:].mean())
        feats["density_std"] = float(dists[:, 1:].std())
    else:
        feats["density"] = 0.05
        feats["density_std"] = 0.0

    # Локальная кривизна: lam3 / sum(lam) в окрестности каждой точки
    if len(seg) >= 10:
        kk = min(15, len(seg) - 1)
        nbrs = NearestNeighbors(n_neighbors=kk + 1).fit(seg)
        _, idx = nbrs.kneighbors(seg)
        curvature = []
        for i in range(len(seg)):
            local = seg[idx[i, 1:]]
            ev = np.sort(np.linalg.eigvalsh(np.cov(local.T)))[::-1]
            ev = np.maximum(ev, 1e-12)
            curvature.append(ev[2] / ev.sum())
        curvature = np.array(curvature)
        feats["curvature_mean"] = float(curvature.mean())
        feats["curvature_std"] = float(curvature.std())
    else:
        feats["curvature_mean"] = 0.0
        feats["curvature_std"] = 0.0

    # Согласованность нормалей: средний |cos| угла между нормалью и нормалями соседей
    if len(seg) >= 10:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(seg)
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamKNN(knn=min(15, len(seg) - 1)))
        normals = np.asarray(pcd.normals)
        kk = min(8, len(seg) - 1)
        nbrs = NearestNeighbors(n_neighbors=kk + 1).fit(seg)
        _, idx = nbrs.kneighbors(seg)
        cos_total, pairs = 0.0, 0
        for i in range(len(seg)):
            for j in idx[i, 1:]:
                cos_total += abs(float(normals[i] @ normals[j]))
                pairs += 1
        feats["normal_consistency"] = cos_total / max(pairs, 1)
    else:
        feats["normal_consistency"] = 0.0

    # Связность: число компонент графа соседей (union-find)
    if len(seg) >= 10:
        tree = cKDTree(seg)
        edges = tree.query_pairs(r=feats["density"] * 2.5)
        parent = list(range(len(seg)))

        def root(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        for a, b in edges:
            ra, rb = root(a), root(b)
            if ra != rb:
                parent[ra] = rb
        feats["n_components"] = len({root(i) for i in range(len(seg))})
    else:
        feats["n_components"] = 1

    return feats

## 5. Определение типа сегмента

Решающее правило по `linearity` / `planarity` / `sphericity` плюс `normal_consistency`.
Согласованность нормалей добавлена к чисто-PCA-порогу, потому что она надёжнее отделяет
гладкие поверхности от зашумлённых фрагментов.

In [ ]:
def infer_segment_type(feats):
    lin = feats["linearity"]
    pla = feats["planarity"]
    sph = feats["sphericity"]
    nc = feats["normal_consistency"]
    curv = feats["curvature_mean"]

    # плоскость: высокая планарность, согласованные нормали, малая кривизна
    if pla > 0.5 and nc > 0.85 and curv < 0.05:
        return "plane"
    # труба / цилиндр: вытянутость по одной оси, нормали согласованы умеренно
    if lin > 0.5 and 0.5 < nc < 0.95:
        return "tube"
    # сфера: изотропность (sphericity не мала, planarity не доминирует),
    # нормали наружу -> умеренная корреляция между соседями
    if sph > 0.25 and pla < 0.5 and lin < 0.5 and 0.4 < nc < 0.85:
        return "sphere"
    return "complex" 

## 6. Подбор алгоритма реконструкции

In [ ]:
def pick_reconstruction(seg_type, feats):
    '''Возвращает (имя_метода, параметры). Параметры зависят от плотности сегмента.'''
    d = feats["density"]
    if seg_type == "plane":
        # плоскости — Alpha Shape (короткие рёбра, нет "раздувания")
        return "alpha_shape", {"alpha": max(d * 2.5, 0.01)}
    if seg_type == "tube":
        # трубчатые — Ball Pivoting с набором радиусов
        return "ball_pivoting", {"radii": [d * 1.5, d * 2.5, d * 4.0]}
    if seg_type == "sphere":
        # сферические — Poisson (хорошо замыкает поверхность)
        return "poisson", {"depth": 8}
    # сложные — Poisson с большей глубиной
    return "poisson", {"depth": 9}

## 7. Реконструкция сегмента

In [ ]:
def build_mesh(seg, method, params):
    '''Применяет выбранный алгоритм. Возвращает open3d.TriangleMesh либо None.'''
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(seg)
    # нормали требуются и Poisson, и Ball Pivoting
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    try:
        pcd.orient_normals_consistent_tangent_plane(k=15)
    except Exception:
        pass

    try:
        if method == "poisson":
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                pcd, depth=params["depth"])
            # отсекаем низкоплотный "раздутый" хвост
            dens = np.asarray(densities)
            if len(dens) > 0:
                mesh.remove_vertices_by_mask(dens < np.quantile(dens, 0.05))
            return mesh
        if method == "alpha_shape":
            return o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
                pcd, params["alpha"])
        if method == "ball_pivoting":
            return o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
                pcd, o3d.utility.DoubleVector(params["radii"]))
    except Exception as err:
        warnings.warn(f"Не удалась реконструкция ({method}): {err}")
    return None

## 8. Метрики качества mesh

Считаем:
- **RMSE** — среднеквадратичная ошибка расстояния от исходных точек до поверхности.
- **Hausdorff (95-й перцентиль)** — устойчивая оценка максимального отклонения.
- **Chamfer (двусторонний)** — учитывает направления точки→mesh и mesh→точки.
- **n_artifacts** — количество несвязных компонент mesh (>1 означает фрагментацию).
- **watertight** — замкнута ли поверхность.

In [ ]:
EMPTY_METRICS = {
    "rmse": float("inf"), "hausdorff_p95": float("inf"), "chamfer": float("inf"),
    "n_artifacts": -1, "watertight": False, "n_vertices": 0, "n_triangles": 0,
}


def score_mesh(source_points, mesh):
    '''Метрики качества для пары (исходные точки, mesh).'''
    out = dict(EMPTY_METRICS)
    if mesh is None or len(mesh.vertices) == 0:
        return out

    n_target = max(int(len(source_points)), 200)
    sampled = mesh.sample_points_uniformly(number_of_points=n_target)
    sampled_pts = np.asarray(sampled.points)
    if len(sampled_pts) == 0:
        return out

    # расстояния точки -> mesh и mesh -> точки
    d_p2m, _ = KDTree(sampled_pts).query(source_points)
    d_m2p, _ = KDTree(source_points).query(sampled_pts)

    out["rmse"] = float(np.sqrt(np.mean(d_p2m ** 2)))
    out["hausdorff_p95"] = float(max(np.quantile(d_p2m, 0.95), np.quantile(d_m2p, 0.95)))
    out["chamfer"] = float(np.mean(d_p2m) + np.mean(d_m2p))

    try:
        _, cluster_n_tri, _ = mesh.cluster_connected_triangles()
        out["n_artifacts"] = int(np.asarray(cluster_n_tri).size)
    except Exception:
        out["n_artifacts"] = -1
    try:
        out["watertight"] = bool(mesh.is_watertight())
    except Exception:
        out["watertight"] = False

    out["n_vertices"] = int(np.asarray(mesh.vertices).shape[0])
    out["n_triangles"] = int(np.asarray(mesh.triangles).shape[0])
    return out

## 9. Объединение в итоговую модель

In [ ]:
def combine_meshes(mesh_list):
    merged = o3d.geometry.TriangleMesh()
    for m in mesh_list:
        if m is not None and len(m.vertices) > 0:
            merged += m
    merged = merged.merge_close_vertices(1e-6)
    merged = merged.remove_degenerate_triangles()
    merged = merged.remove_unreferenced_vertices()
    merged = merged.remove_duplicated_triangles()
    return merged

## 10. Полный pipeline для одного файла

In [ ]:
def run_pipeline(path, save_mesh=True):
    '''Полный проход. Возвращает (mesh, per_segment_rows, summary).'''
    coords, labels = read_ply_cloud(path)
    coords, labels, _, _ = clean_and_normalize(coords, labels)
    segments = segments_by_label(coords, labels, min_points=50)

    meshes, rows = [], []
    for seg_id, seg_pts in segments.items():
        feats = extract_geometry_features(seg_pts)
        seg_type = infer_segment_type(feats)
        method, params = pick_reconstruction(seg_type, feats)
        mesh = build_mesh(seg_pts, method, params)
        metrics = score_mesh(seg_pts, mesh) if mesh else dict(EMPTY_METRICS)

        rows.append({
            "segment": seg_id, "n_points": len(seg_pts),
            "type": seg_type, "method": method,
            "linearity": feats["linearity"], "planarity": feats["planarity"],
            "sphericity": feats["sphericity"],
            "curvature_mean": feats["curvature_mean"],
            "normal_consistency": feats["normal_consistency"],
            "n_components": feats["n_components"],
            **metrics,
        })
        if mesh is not None:
            meshes.append(mesh)

    final = combine_meshes(meshes)
    if save_mesh and len(final.vertices) > 0:
        o3d.io.write_triangle_mesh(
            str(MESHES_DIR / (Path(path).stem + "_recon.ply")), final)

    global_metrics = score_mesh(coords, final)
    summary = {
        "file": str(path), "n_segments": len(segments),
        "global_rmse": global_metrics["rmse"],
        "global_hausdorff": global_metrics["hausdorff_p95"],
        "global_chamfer": global_metrics["chamfer"],
        "global_artifacts": global_metrics["n_artifacts"],
    }
    return final, rows, summary

## 11. Прогон по всей папке

In [ ]:
import csv

cloud_files = sorted(CLOUDS_DIR.glob("*.ply"))
print(f"К обработке: {len(cloud_files)} файлов")

segment_records = []
file_summaries = []
start = time.time()

for i, path in enumerate(cloud_files):
    if i == 0 or (i + 1) % 25 == 0:
        print(f"  [{i + 1}/{len(cloud_files)}] {path.name}  (прошло {time.time() - start:.1f}s)")
    try:
        # mesh сохраняем только для первых 10 файлов
        _, rows, summary = run_pipeline(path, save_mesh=(i < 10))
        for r in rows:
            r["file"] = path.name
        segment_records.extend(rows)
        file_summaries.append(summary)
    except Exception as err:
        print(f"    !!! ошибка на {path.name}: {err}")

print(f"\nГотово. Сегментов всего: {len(segment_records)}. "
      f"Время: {time.time() - start:.1f}s")

csv_path = RESULT_DIR / "segments_report.csv"
if segment_records:
    with open(csv_path, "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(segment_records[0].keys()))
        writer.writeheader()
        writer.writerows(segment_records)
    print("CSV с деталями:", csv_path)

with open(RESULT_DIR / "summaries.json", "w") as fh:
    json.dump(file_summaries, fh, indent=2)

## 12. Сводка по типам сегментов и методам

In [ ]:
type_counts = Counter(r["type"] for r in segment_records)
method_counts = Counter(r["method"] for r in segment_records)
print("Типы сегментов:", dict(type_counts))
print("Методы:        ", dict(method_counts))

rmse_by_type = defaultdict(list)
for r in segment_records:
    if np.isfinite(r["rmse"]):
        rmse_by_type[r["type"]].append(r["rmse"])

print("\nRMSE по типу сегмента:")
for t, values in rmse_by_type.items():
    print(f"  {t:>8s}: mean={np.mean(values):.4f}, "
          f"median={np.median(values):.4f}, N={len(values)}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(type_counts.keys(), type_counts.values(), color="#2563eb")
axes[0].set_title("Сегменты по типам")
axes[0].set_ylabel("Кол-во")
axes[1].bar(method_counts.keys(), method_counts.values(), color="#16a34a")
axes[1].set_title("Выбранные методы")
axes[1].set_ylabel("Кол-во")
plt.tight_layout()
plt.savefig(RESULT_DIR / "type_method_distribution.png", dpi=120)
plt.show()

order = ["plane", "tube", "sphere", "complex"]
box_data = [rmse_by_type[t] for t in order if rmse_by_type[t]]
box_labels = [t for t in order if rmse_by_type[t]]
plt.figure(figsize=(8, 4))
plt.boxplot(box_data, labels=box_labels)
plt.ylabel("RMSE")
plt.title("RMSE по типам сегментов")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / "rmse_by_type.png", dpi=120)
plt.show()

## 13. Сравнение всех методов на одинаковых сегментах

Берём по одному сегменту каждого типа и применяем к ним **все три** алгоритма, чтобы
проверить, оправдано ли адаптивное переключение между Poisson / Alpha / Ball Pivoting.

In [ ]:
def try_all_methods(seg, density):
    scored = {}
    for name, params in [
        ("poisson", {"depth": 8}),
        ("alpha_shape", {"alpha": max(density * 2.5, 0.01)}),
        ("ball_pivoting", {"radii": [density * 1.5, density * 2.5, density * 4.0]}),
    ]:
        scored[name] = score_mesh(seg, build_mesh(seg, name, params))
    return scored


# по одному примеру каждого типа
samples = {}
for r in segment_records:
    if r["type"] not in samples and r["n_points"] > 200:
        samples[r["type"]] = r
    if len(samples) == 4:
        break

comparison = []
for seg_type, r in samples.items():
    coords, labels = read_ply_cloud(CLOUDS_DIR / r["file"])
    coords, labels, _, _ = clean_and_normalize(coords, labels)
    seg_pts = coords[labels == r["segment"]]
    if len(seg_pts) < 50:
        continue
    feats = extract_geometry_features(seg_pts)
    for method, metrics in try_all_methods(seg_pts, feats["density"]).items():
        comparison.append({"type": seg_type, "method": method, **metrics})

print(f'\n{"Type":>8s} | {"Method":>14s} | {"RMSE":>8s} | '
      f'{"Hausdorff":>10s} | {"Chamfer":>8s} | {"Artifacts":>9s}')
print("-" * 80)
for r in comparison:
    rmse = f'{r["rmse"]:.4f}' if np.isfinite(r["rmse"]) else "inf"
    hd = f'{r["hausdorff_p95"]:.4f}' if np.isfinite(r["hausdorff_p95"]) else "inf"
    ch = f'{r["chamfer"]:.4f}' if np.isfinite(r["chamfer"]) else "inf"
    print(f'{r["type"]:>8s} | {r["method"]:>14s} | {rmse:>8s} | '
          f'{hd:>10s} | {ch:>8s} | {r["n_artifacts"]:>9d}')

## 14. Визуализация: исходное облако и реконструкция

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (регистрирует проекцию 3d)

example_path = cloud_files[0]
recon_mesh, _, summary = run_pipeline(example_path, save_mesh=False)
coords, labels = read_ply_cloud(example_path)
coords, labels, _, _ = clean_and_normalize(coords, labels)

fig = plt.figure(figsize=(14, 6))
ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=labels, cmap="tab20", s=1)
ax1.set_title(f"Исходное облако (по сегментам)\n{example_path.name}")
ax1.set_axis_off()

ax2 = fig.add_subplot(122, projection="3d")
if len(recon_mesh.vertices) > 0:
    verts = np.asarray(recon_mesh.vertices)
    tris = np.asarray(recon_mesh.triangles)
    ax2.plot_trisurf(verts[:, 0], verts[:, 1], tris, verts[:, 2],
                     cmap="viridis", alpha=0.85, linewidth=0)
    ax2.set_title(f'Реконструкция\n{summary["n_segments"]} сегментов, '
                  f'RMSE={summary["global_rmse"]:.4f}')
else:
    ax2.set_title("Реконструкция пуста")
ax2.set_axis_off()
plt.tight_layout()
plt.savefig(RESULT_DIR / "recon_example.png", dpi=120)
plt.show()

## 15. Выводы

1. Собран полный pipeline: чтение → предобработка → сегментация по `label` →
   геометрический анализ → классификация → выбор метода → реконструкция → сборка → метрики.
2. Тип сегмента определяется по собственным значениям ковариационной матрицы (PCA)
   и согласованности нормалей — это устойчиво разделяет плоскости / трубы / сферы / сложные формы.
3. Под каждый тип подобран свой алгоритм:
   - **plane** → Alpha Shape (аккуратно обрезает границы),
   - **tube** → Ball Pivoting (несколько радиусов под разную плотность),
   - **sphere/complex** → Poisson (замкнутая поверхность).
4. Кроме RMSE считаются Hausdorff-p95, Chamfer и число артефактов — это даёт более полную
   картину качества, чем одна метрика.
5. Сравнение всех методов на одинаковых сегментах (раздел 13) подтверждает, что адаптивный
   выбор алгоритма выигрывает у единого метода на весь датасет.